In [87]:
import pandas as pd
import os

path = r"C:\Users\yashr\.cache\kagglehub\datasets\martj42\international-football-results-from-1872-to-2017\versions\138"

results = pd.read_csv(os.path.join(path, "results.csv"))

results["date"] = pd.to_datetime(results["date"])

print("Total matches:", len(results))
print("First match:", results["date"].min().date())
print("Latest match:", results["date"].max().date())
print("Unique teams:", len(set(results["home_team"]) | set(results["away_team"])))
print("\nMost recent 5 matches:")
print(results[["date", "home_team", "away_team", "home_score", "away_score"]].tail())

Total matches: 49945
First match: 1872-11-30
Latest match: 2026-10-06
Unique teams: 337

Most recent 5 matches:
            date                 home_team             away_team  home_score  \
49940 2026-10-06                    Belize         French Guiana           0   
49941 2026-10-06  Turks and Caicos Islands            Montserrat           1   
49942 2026-10-06       Antigua and Barbuda                 Aruba           3   
49943 2026-10-06              Saint Martin               Bahamas           6   
49944 2026-10-06              Saudi Arabia  United Arab Emirates           2   

       away_score  
49940           1  
49941           4  
49942           2  
49943           0  
49944           0  


In [88]:
matches = results.copy()

matches["date"] = pd.to_datetime(matches["date"])

matches = matches.sort_values(
    ["date"]
).reset_index(drop=True)

matches["match_id"] = range(len(matches))

print("Total matches:", len(matches))
print("Dates sorted:", matches["date"].is_monotonic_increasing)
print("Duplicate match IDs:", matches["match_id"].duplicated().sum())

print("\nFirst 5 matches:")
print(
    matches[
        ["match_id", "date", "home_team", "away_team",
         "home_score", "away_score", "neutral"]
    ].head().to_string(index=False)
)

print("\nLast 5 matches:")
print(
    matches[
        ["match_id", "date", "home_team", "away_team",
         "home_score", "away_score", "neutral"]
    ].tail().to_string(index=False)
)

Total matches: 49945
Dates sorted: True
Duplicate match IDs: 0

First 5 matches:
 match_id       date home_team away_team  home_score  away_score  neutral
        0 1872-11-30  Scotland   England           0           0    False
        1 1873-03-08   England  Scotland           4           2    False
        2 1874-03-07  Scotland   England           2           1    False
        3 1875-03-06   England  Scotland           2           2    False
        4 1876-03-04  Scotland   England           3           0    False

Last 5 matches:
 match_id       date    home_team            away_team  home_score  away_score  neutral
    49940 2026-10-06        China           Tajikistan           0           1    False
    49941 2026-10-06    Argentina                Benin           3           0    False
    49942 2026-10-06      Algeria                Niger           2           1    False
    49943 2026-10-06     Scotland             Slovenia           1           2    False
    49944 2026-10-

In [89]:
def expected_score(rating_a, rating_b):
    """Expected result for team A against team B."""
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))


def actual_score(goals_a, goals_b):
    """Actual match result from team A's perspective."""
    if goals_a > goals_b:
        return 1.0
    elif goals_a < goals_b:
        return 0.0
    return 0.5


def update_elo(rating, expected, actual, k=20):
    """Calculate a team's rating after one match."""
    return rating + k * (actual - expected)

In [90]:
from collections import defaultdict
import pandas as pd


def calculate_elo_ratings(matches, initial_rating=1500, k=20):
    ratings = defaultdict(lambda: initial_rating)
    history = []

    matches = matches.sort_values("date").copy()

    for date, daily_matches in matches.groupby("date", sort=True):
        # Every match on this date uses the same starting ratings.
        daily_updates = defaultdict(float)
        daily_records = []

        for _, match in daily_matches.iterrows():
            home = match["home_team"]
            away = match["away_team"]

            home_rating = ratings[home]
            away_rating = ratings[away]

            expected_home = expected_score(home_rating, away_rating)
            expected_away = 1 - expected_home

            actual_home = actual_score(
                match["home_score"], match["away_score"]
            )
            actual_away = 1 - actual_home

            home_change = k * (actual_home - expected_home)
            away_change = k * (actual_away - expected_away)

            daily_updates[home] += home_change
            daily_updates[away] += away_change

            daily_records.append({
                "date": date,
                "home_team": home,
                "away_team": away,
                "home_score": match["home_score"],
                "away_score": match["away_score"],
                "home_rating_before": home_rating,
                "away_rating_before": away_rating,
                "expected_home": expected_home,
                "actual_home": actual_home,
                "home_elo_change": home_change,
                "away_elo_change": away_change,
                "tournament": match["tournament"],
                "neutral": match["neutral"],
            })

        # Apply updates only after processing every match on this date.
        for team, change in daily_updates.items():
            ratings[team] += change

        history.extend(daily_records)

    return ratings, pd.DataFrame(history)


final_ratings, elo_history = calculate_elo_ratings(matches)

print("Matches processed:", len(elo_history))
print("Teams rated:", len(final_ratings))
print("Dates processed:", elo_history["date"].nunique())

print("\nTop 15 teams by final Elo:")
print(
    pd.Series(final_ratings)
    .sort_values(ascending=False)
    .head(15)
    .round(1)
    .to_string()
)

Matches processed: 49945
Teams rated: 337
Dates processed: 16535

Top 15 teams by final Elo:
Spain          2039.6
Argentina      2014.6
France         1967.3
England        1936.1
Portugal       1925.3
Brazil         1921.2
Netherlands    1887.6
Colombia       1883.7
Germany        1866.7
Japan          1862.2
Italy          1861.5
Belgium        1861.1
Morocco        1860.9
Switzerland    1832.5
Croatia        1820.2


In [91]:
final_ratings_k50, elo_history_k50 = calculate_elo_ratings(
    matches,
    k=50
)

print("Elo history shape:", elo_history_k50.shape)
print("Missing home ratings:",
      elo_history_k50["home_rating_before"].isna().sum())
print("Missing away ratings:",
      elo_history_k50["away_rating_before"].isna().sum())

Elo history shape: (49945, 13)
Missing home ratings: 0
Missing away ratings: 0


In [92]:
goal_model_data = elo_history_k50.copy()

goal_model_data["elo_gap"] = (
    goal_model_data["home_rating_before"]
    - goal_model_data["away_rating_before"]
)

print("Dataset shape:", goal_model_data.shape)
print("\nColumns:")
print(goal_model_data.columns.tolist())

print("\nMissing values:")
print(
    goal_model_data[
        ["home_rating_before", "away_rating_before", "elo_gap"]
    ].isna().sum()
)

print("\nSample:")
print(goal_model_data.head().to_string(index=False))

Dataset shape: (49945, 14)

Columns:
['date', 'home_team', 'away_team', 'home_score', 'away_score', 'home_rating_before', 'away_rating_before', 'expected_home', 'actual_home', 'home_elo_change', 'away_elo_change', 'tournament', 'neutral', 'elo_gap']

Missing values:
home_rating_before    0
away_rating_before    0
elo_gap               0
dtype: int64

Sample:
      date home_team away_team  home_score  away_score  home_rating_before  away_rating_before  expected_home  actual_home  home_elo_change  away_elo_change tournament  neutral    elo_gap
1872-11-30  Scotland   England           0           0         1500.000000         1500.000000       0.500000          0.5         0.000000         0.000000   Friendly    False   0.000000
1873-03-08   England  Scotland           4           2         1500.000000         1500.000000       0.500000          1.0        25.000000       -25.000000   Friendly    False   0.000000
1874-03-07  Scotland   England           2           1         1475.000000 

In [93]:
# Modern international matches for initial goal-model development

recent_data = goal_model_data[
    goal_model_data["date"] >= "2010-01-01"
].copy()

recent_data["total_goals"] = (
    recent_data["home_score"] + recent_data["away_score"]
)

print("Matches:", len(recent_data))
print("Date range:", recent_data["date"].min(), "to", recent_data["date"].max())
print("Average home goals:", round(recent_data["home_score"].mean(), 3))
print("Average away goals:", round(recent_data["away_score"].mean(), 3))
print("Average total goals:", round(recent_data["total_goals"].mean(), 3))

print("\nTournament types:")
print(recent_data["tournament"].value_counts().head(15).to_string())

Matches: 16339
Date range: 2010-01-02 00:00:00 to 2026-10-06 00:00:00
Average home goals: 1.61
Average away goals: 1.123
Average total goals: 2.732

Tournament types:
tournament
Friendly                                5084
FIFA World Cup qualification            3422
African Cup of Nations qualification    1112
UEFA Euro qualification                 1016
UEFA Nations League                      762
CONCACAF Nations League                  495
African Cup of Nations                   365
FIFA World Cup                           360
AFC Asian Cup qualification              269
Gold Cup                                 225
COSAFA Cup                               201
UEFA Euro                                184
Copa América                             170
AFC Asian Cup                            166
AFF Championship                         150


In [94]:
# Chronological split: no random shuffling

train_goals = recent_data[
    recent_data["date"] < "2019-01-01"
].copy()

test_goals = recent_data[
    recent_data["date"] >= "2019-01-01"
].copy()

print("Training period:")
print(train_goals["date"].min(), "to", train_goals["date"].max())
print("Training matches:", len(train_goals))
print("Training average goals:",
      round((train_goals["home_score"] + train_goals["away_score"]).mean(), 3))

print("\nTesting period:")
print(test_goals["date"].min(), "to", test_goals["date"].max())
print("Testing matches:", len(test_goals))
print("Testing average goals:",
      round((test_goals["home_score"] + test_goals["away_score"]).mean(), 3))

print("\nChronological split valid:",
      train_goals["date"].max() < test_goals["date"].min())

Training period:
2010-01-02 00:00:00 to 2018-12-31 00:00:00
Training matches: 8644
Training average goals: 2.717

Testing period:
2019-01-02 00:00:00 to 2026-10-06 00:00:00
Testing matches: 7695
Testing average goals: 2.75

Chronological split valid: True


In [95]:
import numpy as np
import statsmodels.api as sm
from math import lgamma

# Prepare features using pre-match information only
features = ["home_rating_before", "away_rating_before", "neutral"]

X_train = train_goals[features].copy()
X_test = test_goals[features].copy()

# Convert neutral status to numeric
X_train["neutral"] = X_train["neutral"].astype(int)
X_test["neutral"] = X_test["neutral"].astype(int)

# Add intercept
X_train = sm.add_constant(X_train, has_constant="add")
X_test = sm.add_constant(X_test, has_constant="add")

# Fit separate Poisson models on training data
home_model = sm.GLM(
    train_goals["home_score"],
    X_train,
    family=sm.families.Poisson()
).fit()

away_model = sm.GLM(
    train_goals["away_score"],
    X_train,
    family=sm.families.Poisson()
).fit()

# Predict test-period expected goals
home_lambda = np.maximum(home_model.predict(X_test), 1e-9)
away_lambda = np.maximum(away_model.predict(X_test), 1e-9)

# Evaluate Poisson negative log-likelihood
def poisson_loss_array(actual, predicted):
    actual = np.asarray(actual)
    predicted = np.maximum(np.asarray(predicted), 1e-9)
    log_prob = (
        actual * np.log(predicted)
        - predicted
        - np.array([lgamma(int(x) + 1) for x in actual])
    )
    return -log_prob.mean()

home_loss_elo = poisson_loss_array(test_goals["home_score"], home_lambda)
away_loss_elo = poisson_loss_array(test_goals["away_score"], away_lambda)

print("Home-goal model loss:", round(home_loss_elo, 4))
print("Away-goal model loss:", round(away_loss_elo, 4))
print("Combined mean loss:", round((home_loss_elo + away_loss_elo) / 2, 4))
print("\nHome model coefficients:")
print(home_model.params.round(5))
print("\nAway model coefficients:")
print(away_model.params.round(5))

Home-goal model loss: 1.5519
Away-goal model loss: 1.3269
Combined mean loss: 1.4394

Home model coefficients:
const                 1.00779
home_rating_before    0.00181
away_rating_before   -0.00222
neutral              -0.06008
dtype: float64

Away model coefficients:
const                 0.10601
home_rating_before   -0.00222
away_rating_before    0.00212
neutral               0.30488
dtype: float64


In [96]:
import numpy as np

def simulate_match(home_lambda, away_lambda, rng=None):
    """
    Simulate a match score using Poisson-distributed goals.
    Returns (home_goals, away_goals).
    """
    if rng is None:
        rng = np.random.default_rng()

    home_goals = rng.poisson(max(float(home_lambda), 1e-9))
    away_goals = rng.poisson(max(float(away_lambda), 1e-9))

    return int(home_goals), int(away_goals)


# Quick test
rng = np.random.default_rng(42)

for _ in range(5):
    print(simulate_match(1.5, 1.1, rng))

(3, 0)
(3, 1)
(3, 0)
(1, 3)
(3, 1)


In [97]:

def simulate_knockout_match(
    home_lambda,
    away_lambda,
    home_rating,
    away_rating,
    rng=None
):
    if rng is None:
        rng = np.random.default_rng()

    # First 90 minutes
    home_goals, away_goals = simulate_match(
        home_lambda, away_lambda, rng
    )

    if home_goals != away_goals:
        winner = "home" if home_goals > away_goals else "away"
        return home_goals, away_goals, "90 minutes", winner

    # Extra time (approximation: one-third of 90-minute goal rates)
    extra_home, extra_away = simulate_match(
        home_lambda / 3, away_lambda / 3, rng
    )
    home_goals += extra_home
    away_goals += extra_away

    if home_goals != away_goals:
        winner = "home" if home_goals > away_goals else "away"
        return home_goals, away_goals, "Extra time", winner

    # Penalties decide the winner without changing the match score
    home_win_probability = 1 / (
        1 + 10 ** ((away_rating - home_rating) / 400)
    )

    winner = "home" if rng.random() < home_win_probability else "away"
    return home_goals, away_goals, "Penalties", winner


print("Knockout match simulator updated.")


Knockout match simulator updated.


In [98]:
# Inspect the 2026 World Cup teams and match structure

wc_2026 = results[
    (results["tournament"] == "FIFA World Cup") &
    (results["date"].dt.year == 2026)
].copy()

teams_2026 = sorted(
    set(wc_2026["home_team"]) |
    set(wc_2026["away_team"])
)

print("2026 World Cup matches in dataset:", len(wc_2026))
print("Unique teams appearing in matches:", len(teams_2026))
print("\nTeams:")
print(teams_2026)

2026 World Cup matches in dataset: 104
Unique teams appearing in matches: 48

Teams:
['Algeria', 'Argentina', 'Australia', 'Austria', 'Belgium', 'Bosnia and Herzegovina', 'Brazil', 'Canada', 'Cape Verde', 'Colombia', 'Croatia', 'Curaçao', 'Czech Republic', 'DR Congo', 'Ecuador', 'Egypt', 'England', 'France', 'Germany', 'Ghana', 'Haiti', 'Iran', 'Iraq', 'Ivory Coast', 'Japan', 'Jordan', 'Mexico', 'Morocco', 'Netherlands', 'New Zealand', 'Norway', 'Panama', 'Paraguay', 'Portugal', 'Qatar', 'Saudi Arabia', 'Scotland', 'Senegal', 'South Africa', 'South Korea', 'Spain', 'Sweden', 'Switzerland', 'Tunisia', 'Turkey', 'United States', 'Uruguay', 'Uzbekistan']


In [99]:
# Official 2026 World Cup group draw
world_cup_groups = {
    "A": ["Mexico", "South Africa", "South Korea", "Czech Republic"],
    "B": ["Canada", "Bosnia and Herzegovina", "Qatar", "Switzerland"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["United States", "Paraguay", "Australia", "Turkey"],
    "E": ["Germany", "Curaçao", "Ecuador", "Ivory Coast"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Iraq", "Norway"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "Colombia", "Uzbekistan", "DR Congo"],
    "L": ["England", "Croatia", "Ghana", "Panama"]
}

# Validate the group draw
group_teams = [
    team
    for group in world_cup_groups.values()
    for team in group
]

print("Number of groups:", len(world_cup_groups))
print("Teams assigned:", len(group_teams))
print("Unique teams:", len(set(group_teams)))
print("Teams missing from groups:", sorted(set(teams_2026) - set(group_teams)))
print("Unknown team names:", sorted(set(group_teams) - set(teams_2026)))

assert len(world_cup_groups) == 12
assert all(len(group) == 4 for group in world_cup_groups.values())
assert len(set(group_teams)) == 48
assert set(group_teams) == set(teams_2026)

print("\nSUCCESS: All 48 teams assigned exactly once.")

Number of groups: 12
Teams assigned: 48
Unique teams: 48
Teams missing from groups: []
Unknown team names: []

SUCCESS: All 48 teams assigned exactly once.


In [100]:
def get_goal_rates(team_a, team_b, ratings):
    """
    Return expected goals for team_a and team_b in a neutral matchup.
    Uses the trained Poisson models and pre-tournament Elo ratings.
    """
    rating_a = ratings.get(team_a, 1500)
    rating_b = ratings.get(team_b, 1500)

    # Team A listed as home
    matchup_a_home = pd.DataFrame({
        "const": [1.0],
        "home_rating_before": [rating_a],
        "away_rating_before": [rating_b],
        "neutral": [1]
    })

    # Team B listed as home
    matchup_b_home = pd.DataFrame({
        "const": [1.0],
        "home_rating_before": [rating_b],
        "away_rating_before": [rating_a],
        "neutral": [1]
    })

    # Average both orientations to avoid arbitrary home/away ordering
    a_goals_first = float(home_model.predict(matchup_a_home).iloc[0])
    b_goals_first = float(away_model.predict(matchup_a_home).iloc[0])

    b_goals_second = float(home_model.predict(matchup_b_home).iloc[0])
    a_goals_second = float(away_model.predict(matchup_b_home).iloc[0])

    lambda_a = max((a_goals_first + a_goals_second) / 2, 0.05)
    lambda_b = max((b_goals_first + b_goals_second) / 2, 0.05)

    return lambda_a, lambda_b

In [101]:
# Reconstruct exact Elo ratings immediately before the tournament

from collections import defaultdict

wc_2026_start = wc_2026["date"].min()

pre_wc_matches = elo_history_k50[
    elo_history_k50["date"] < wc_2026_start
].copy()

pre_wc_ratings = defaultdict(lambda: 1500.0)

for date, daily_matches in pre_wc_matches.groupby("date", sort=True):
    daily_updates = defaultdict(float)

    for _, row in daily_matches.iterrows():
        daily_updates[row["home_team"]] += row["home_elo_change"]
        daily_updates[row["away_team"]] += row["away_elo_change"]

    for team, change in daily_updates.items():
        pre_wc_ratings[team] += change

pre_wc_ratings = dict(pre_wc_ratings)

print("Snapshot date:", wc_2026_start.date())
print("Rated teams:", len(pre_wc_ratings))

print("\nTop 10 pre-tournament Elo ratings:")
print(
    sorted(pre_wc_ratings.items(), key=lambda x: x[1], reverse=True)[:10]
)

print("\nSpain vs Argentina expected goals:")
print(get_goal_rates("Spain", "Argentina", pre_wc_ratings))

Snapshot date: 2026-06-11
Rated teams: 337

Top 10 pre-tournament Elo ratings:
[('Spain', 2124.642587307452), ('Argentina', 2104.31222259818), ('France', 2046.7242810511332), ('Portugal', 2023.4152157015697), ('Germany', 2020.6136254348146), ('England', 1997.7207984684062), ('Brazil', 1990.8630673510484), ('Japan', 1981.8808813168841), ('Morocco', 1981.1760646981918), ('Colombia', 1976.691840425642)]

Spain vs Argentina expected goals:
(1.1910067309446493, 1.093641655617247)


In [102]:
# Precompute expected goals for all possible team matchups
from itertools import combinations

goal_rates = {}

for team_a, team_b in combinations(teams_2026, 2):
    rates_a, rates_b = get_goal_rates(
        team_a, team_b, pre_wc_ratings
    )

    goal_rates[(team_a, team_b)] = (rates_a, rates_b)
    goal_rates[(team_b, team_a)] = (rates_b, rates_a)


def get_pair_rates(team_a, team_b):
    return goal_rates[(team_a, team_b)]


# Adapt the group simulator to use matchup-specific goal rates
def simulate_group_stage(groups, ratings, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    all_standings = []
    match_results = []

    for group_name, teams in groups.items():
        table = {
            team: {
                "team": team,
                "group": group_name,
                "played": 0,
                "wins": 0,
                "draws": 0,
                "losses": 0,
                "goals_for": 0,
                "goals_against": 0,
                "points": 0
            }
            for team in teams
        }

        for team_a, team_b in combinations(teams, 2):
            lambda_a, lambda_b = get_pair_rates(team_a, team_b)
            goals_a, goals_b = simulate_match(
                lambda_a, lambda_b, rng
            )

            table[team_a]["played"] += 1
            table[team_b]["played"] += 1
            table[team_a]["goals_for"] += goals_a
            table[team_a]["goals_against"] += goals_b
            table[team_b]["goals_for"] += goals_b
            table[team_b]["goals_against"] += goals_a

            if goals_a > goals_b:
                table[team_a]["wins"] += 1
                table[team_b]["losses"] += 1
                table[team_a]["points"] += 3
            elif goals_b > goals_a:
                table[team_b]["wins"] += 1
                table[team_a]["losses"] += 1
                table[team_b]["points"] += 3
            else:
                table[team_a]["draws"] += 1
                table[team_b]["draws"] += 1
                table[team_a]["points"] += 1
                table[team_b]["points"] += 1

            match_results.append({
                "group": group_name,
                "team_a": team_a,
                "team_b": team_b,
                "goals_a": goals_a,
                "goals_b": goals_b
            })

        group_table = pd.DataFrame(table.values())
        group_table["goal_difference"] = (
            group_table["goals_for"] - group_table["goals_against"]
        )

        group_table = group_table.sort_values(
            ["points", "goal_difference", "goals_for", "team"],
            ascending=[False, False, False, True]
        ).reset_index(drop=True)

        group_table["position"] = range(1, 5)
        all_standings.append(group_table)

    return pd.concat(all_standings, ignore_index=True), match_results


# Run one tournament
rng = np.random.default_rng(42)

standings, group_match_results = simulate_group_stage(
    world_cup_groups, pre_wc_ratings, rng
)

print("Group matches simulated:", len(group_match_results))
print("Teams in standings:", len(standings))
print("Matches per team:")
print(standings["played"].value_counts().to_dict())

display(
    standings.sort_values(["group", "position"])[
        ["group", "position", "team", "played", "wins",
         "draws", "losses", "goals_for", "goals_against",
         "goal_difference", "points"]
    ]
)

Group matches simulated: 72
Teams in standings: 48
Matches per team:
{3: 48}


,group,position,team,played,wins,draws,losses,goals_for,goals_against,goal_difference,points
0,A,1,South Korea,3,3,0,0,10,4,6,9
1,A,2,Mexico,3,2,0,1,6,6,0,6
2,A,3,South Africa,3,1,0,2,6,8,-2,3
3,A,4,Czech Republic,3,0,0,3,1,5,-4,0
4,B,1,Switzerland,3,2,1,0,7,1,6,7
5,B,2,Canada,3,1,1,1,5,4,1,4
6,B,3,Qatar,3,1,0,2,4,10,-6,3
7,B,4,Bosnia and Herzegovina,3,0,2,1,3,4,-1,2
8,C,1,Brazil,3,2,1,0,3,1,2,7
9,C,2,Morocco,3,2,0,1,5,1,4,6


In [103]:
def select_qualifiers(standings):
    qualified = []

    for group_name, group_table in standings.groupby("group"):
        group_table = group_table.sort_values(
            ["points", "goal_difference", "goals_for", "team"],
            ascending=[False, False, False, True]
        )

        # Top two from each group
        qualified.extend(group_table.head(2).to_dict("records"))

        # Third-place team enters the best-third comparison
        third_place = group_table.iloc[2].to_dict()
        qualified.append({
            **third_place,
            "is_third_place": True
        })

    # Keep all group winners and runners-up
    top_two = [
        team for team in qualified
        if team["position"] in (1, 2)
    ]

    # Select best 8 third-place teams
    third_place_teams = [
        team for team in qualified
        if team.get("is_third_place", False)
    ]

    third_place_teams = sorted(
        third_place_teams,
        key=lambda x: (
            x["points"],
            x["goal_difference"],
            x["goals_for"],
            x["team"]
        ),
        reverse=True
    )

    best_eight = third_place_teams[:8]

    return pd.DataFrame(top_two + best_eight)


qualifiers = select_qualifiers(standings)

print("Qualified teams:", len(qualifiers))
print("Group winners:", (qualifiers["position"] == 1).sum())
print("Runners-up:", (qualifiers["position"] == 2).sum())
print("Third-place qualifiers:", qualifiers["position"].eq(3).sum())

display(
    qualifiers.sort_values(["group", "position"])[
        ["group", "position", "team", "points",
         "goal_difference", "goals_for"]
    ]
)

assert len(qualifiers) == 32

Qualified teams: 32
Group winners: 12
Runners-up: 12
Third-place qualifiers: 8


,group,position,team,points,goal_difference,goals_for
0,A,1,South Korea,9,6,10
1,A,2,Mexico,6,0,6
30,A,3,South Africa,3,-2,6
2,B,1,Switzerland,7,6,7
3,B,2,Canada,4,1,5
4,C,1,Brazil,7,2,3
5,C,2,Morocco,6,4,5
25,C,3,Scotland,4,1,2
6,D,1,Turkey,9,6,6
7,D,2,Paraguay,6,2,5


In [104]:
import re
import requests
from itertools import combinations

url = "https://raw.githubusercontent.com/manganite/wm2026/main/thirdPlaceAssignments.mjs"

response = requests.get(url, timeout=30)
response.raise_for_status()
source = response.text

winners_match = re.search(
    r"ANNEX_C_WINNERS\s*=\s*(\[[^\]]+\])",
    source
)
rows_match = re.search(
    r"ANNEX_C_ROWS\s*=\s*\[(.*?)\]",
    source,
    re.DOTALL
)

assert winners_match and rows_match, "Could not parse the official allocation table."

winner_groups = re.findall(r'"([A-L])"', winners_match.group(1))
allocation_rows = re.findall(r'"([A-L]{8})"', rows_match.group(1))

assert len(winner_groups) == 8
assert len(allocation_rows) == 495

third_place_allocation = {}

for row in allocation_rows:
    # The eight groups represented by this row
    group_set = "".join(sorted(row))

    # Match each assigned third-place group to its winner's Round of 32 match
    third_place_allocation[group_set] = dict(
        zip(winner_groups, row)
    )

assert len(third_place_allocation) == 495

print("Official allocation rows:", len(allocation_rows))
print("Unique combinations:", len(third_place_allocation))
print("Winner-group columns:", winner_groups)
print("Table loaded and validated.")

Official allocation rows: 495
Unique combinations: 495
Winner-group columns: ['A', 'B', 'D', 'E', 'G', 'I', 'K', 'L']
Table loaded and validated.


In [105]:
round_of_32_template = [
    (73, "R_A", "R_B"),
    (74, "W_E", "T_AB CDF".replace(" ", "")),
    (75, "W_F", "R_C"),
    (76, "W_C", "R_F"),
    (77, "W_I", "T_CDFGH"),
    (78, "R_E", "R_I"),
    (79, "W_A", "T_CEFHI"),
    (80, "W_L", "T_EHIJK"),
    (81, "W_D", "T_BEFIJ"),
    (82, "W_G", "T_AEHIJ"),
    (83, "R_K", "R_L"),
    (84, "W_H", "R_J"),
    (85, "W_B", "T_EFGIJ"),
    (86, "W_J", "R_H"),
    (87, "W_K", "T_DEIJL"),
    (88, "R_D", "R_G"),
]

# Eligible third-place groups for each third-place slot
third_slot_eligibility = {
    "T_AB CDF".replace(" ", ""): set("ABCDF"),
    "T_CDFGH": set("CDFGH"),
    "T_CEFHI": set("CEFHI"),
    "T_EHIJK": set("EHIJK"),
    "T_BEFIJ": set("BEFIJ"),
    "T_AEHIJ": set("AEHIJ"),
    "T_EFGIJ": set("EFGIJ"),
    "T_DEIJL": set("DEIJL"),
}

print("Round of 32 fixtures:", len(round_of_32_template))
print("Third-place slots:", len(third_slot_eligibility))
print("Allocation combinations:", len(third_place_allocation))

Round of 32 fixtures: 16
Third-place slots: 8
Allocation combinations: 495


In [106]:
def assign_third_place_slots(qualifiers):
    qualifiers = qualifiers.copy()

    # Identify the third-place qualifiers and their groups
    third = qualifiers[qualifiers["position"] == 3].copy()

    if len(third) != 8:
        raise ValueError(f"Expected 8 third-place qualifiers, got {len(third)}")

    # Key must be the sorted group letters, e.g. 'EFGHIJKL'
    group_key = "".join(sorted(third["group"].astype(str).tolist()))

    if group_key not in third_place_allocation:
        raise ValueError(f"No official allocation row for {group_key}")

    # Official mapping: winner-group letter -> assigned third-place group
    allocation = third_place_allocation[group_key]

    # Group letter -> qualifying third-place team record
    third_by_group = {
        str(row["group"]): row.to_dict()
        for _, row in third.iterrows()
    }

    assigned = {}
    for slot, third_group in allocation.items():
        assigned[f"T_{slot}"] = third_by_group[third_group]

    return assigned


print("Allocation function created.")

Allocation function created.


In [107]:
def build_round_of_32(qualifiers):
    team_by_group_position = {
        (str(row["group"]), int(row["position"])): row.to_dict()
        for _, row in qualifiers.iterrows()
    }

    third_slots = assign_third_place_slots(qualifiers)

    # The official allocation maps winner-group letters to third-place groups.
    # Keep the actual third-place team records indexed by group.
    third_by_group = {
        str(team["group"]): team
        for team in third_slots.values()
    }

    # Extract all eight third-place fixture slots in bracket order.
    fixture_third_slots = []
    for _, slot_a, slot_b in round_of_32_template:
        for slot in (slot_a, slot_b):
            if slot.startswith("T_") and slot not in fixture_third_slots:
                fixture_third_slots.append(slot)

    # Find a complete assignment, rather than greedily consuming teams.
    eligible = {
        slot: set(third_slot_eligibility[slot])
        for slot in fixture_third_slots
    }

    assignment = {}

    def backtrack(remaining_slots, remaining_groups):
        if not remaining_slots:
            return True

        # Assign the most constrained slot first.
        slot = min(
            remaining_slots,
            key=lambda s: len(eligible[s] & remaining_groups)
        )
        candidates = sorted(eligible[slot] & remaining_groups)

        for group in candidates:
            assignment[slot] = group
            if backtrack(
                [s for s in remaining_slots if s != slot],
                remaining_groups - {group}
            ):
                return True
            assignment.pop(slot)

        return False

    if not backtrack(fixture_third_slots, set(third_by_group)):
        raise ValueError(
            "No complete compatible assignment exists for this combination."
        )

    def resolve_slot(slot):
        if slot.startswith("W_"):
            return team_by_group_position[(slot[2:], 1)]
        if slot.startswith("R_"):
            return team_by_group_position[(slot[2:], 2)]
        if slot.startswith("T_"):
            return third_by_group[assignment[slot]]
        raise ValueError(f"Unknown slot: {slot}")

    fixtures = []
    for match_id, slot_a, slot_b in round_of_32_template:
        team_a = resolve_slot(slot_a)
        team_b = resolve_slot(slot_b)
        fixtures.append({
            "match_id": match_id,
            "team_a": team_a["team"],
            "group_a": team_a["group"],
            "team_b": team_b["team"],
            "group_b": team_b["group"],
        })

    return pd.DataFrame(fixtures)

In [108]:
round_of_16_template = [
    (89, 74, 77),
    (90, 73, 75),
    (91, 76, 78),
    (92, 79, 80),
    (93, 83, 84),
    (94, 81, 82),
    (95, 86, 88),
    (96, 85, 87),
]

quarterfinal_template = [
    (97, 89, 90),
    (98, 93, 94),
    (99, 91, 92),
    (100, 95, 96),
]

semifinal_template = [(101, 97, 98), (102, 99, 100)]
final_template = [(104, 101, 102)]


def simulate_fixed_knockout(qualifiers, ratings, rng):
    r32 = build_round_of_32(qualifiers)
    match_log = []
    winners_by_id = {}

    def play(match_id, team_a, team_b, round_name):
        lambda_a, lambda_b = get_pair_rates(team_a, team_b)
        goals_a, goals_b, decided_by, winner_side = simulate_knockout_match(
            lambda_a, lambda_b,
            ratings.get(team_a, 1500),
            ratings.get(team_b, 1500),
            rng
        )
        winner = team_a if winner_side == "home" else team_b
        winners_by_id[match_id] = winner
        match_log.append({
            "match_id": match_id,
            "round": round_name,
            "team_a": team_a,
            "team_b": team_b,
            "goals_a": goals_a,
            "goals_b": goals_b,
            "winner": winner,
            "decided_by": decided_by,
        })

    # Round of 32
    for _, row in r32.iterrows():
        play(int(row["match_id"]), row["team_a"], row["team_b"], "Round of 32")

    # Later rounds advance through official match IDs
    for match_id, prev_a, prev_b in round_of_16_template:
        play(match_id, winners_by_id[prev_a], winners_by_id[prev_b], "Round of 16")

    for match_id, prev_a, prev_b in quarterfinal_template:
        play(match_id, winners_by_id[prev_a], winners_by_id[prev_b], "Quarter-finals")

    for match_id, prev_a, prev_b in semifinal_template:
        play(match_id, winners_by_id[prev_a], winners_by_id[prev_b], "Semi-finals")

    match_a, match_b = final_template[0][1:]
    play(104, winners_by_id[match_a], winners_by_id[match_b], "Final")

    return {
        "champion": winners_by_id[104],
        "match_log": match_log,
    }

In [109]:
from collections import Counter

def run_monte_carlo(n_simulations=1000, seed=2026):
    rng = np.random.default_rng(seed)

    champion_counts = Counter()
    round_counts = {
    "Qualified": Counter(),
    "Round of 32": Counter(),
    "Round of 16": Counter(),
    "Quarter-finals": Counter(),
    "Semi-finals": Counter(),
    "Final": Counter(),
    "Champion": Counter(),
}

    for _ in range(n_simulations):
        standings, _ = simulate_group_stage(
            world_cup_groups, pre_wc_ratings, rng
        )
        qualifiers = select_qualifiers(standings)

        for team in qualifiers["team"]:
            round_counts["Qualified"][team] += 1

        result = simulate_fixed_knockout(
            qualifiers, pre_wc_ratings, rng
        )
        champion_counts[result["champion"]] += 1

        for match in result["match_log"]:
            round_name = match["round"]
            round_counts[round_name][match["team_a"]] += 1
            round_counts[round_name][match["team_b"]] += 1

        # Teams reaching later rounds are the participants in those matches.
        # Champion is counted separately below.
        for team in [result["champion"]]:
            round_counts["Champion"][team] += 1

    summary = pd.DataFrame({
        "team": sorted(set(round_counts["Qualified"]) |
                       set(champion_counts))
    })

    summary["qualification_pct"] = summary["team"].map(
        lambda t: 100 * round_counts["Qualified"][t] / n_simulations
    )
    summary["round_of_16_pct"] = summary["team"].map(
        lambda t: 100 * round_counts["Round of 16"][t] / n_simulations
    )
    summary["quarterfinal_pct"] = summary["team"].map(
        lambda t: 100 * round_counts["Quarter-finals"][t] / n_simulations
    )
    summary["semifinal_pct"] = summary["team"].map(
        lambda t: 100 * round_counts["Semi-finals"][t] / n_simulations
    )
    summary["final_pct"] = summary["team"].map(
        lambda t: 100 * round_counts["Final"][t] / n_simulations
    )
    summary["champion_pct"] = summary["team"].map(
        lambda t: 100 * champion_counts[t] / n_simulations
    )

    return summary.sort_values("champion_pct", ascending=False).reset_index(drop=True)


mc_results = run_monte_carlo(n_simulations=5000, seed=2026)

display(mc_results.head(15))
print("\nChampion probabilities sum to:",
      round(mc_results["champion_pct"].sum(), 2), "%")

,team,qualification_pct,round_of_16_pct,quarterfinal_pct,semifinal_pct,final_pct,champion_pct
0,Spain,99.64,73.04,55.52,43.74,31.00,19.80
1,Argentina,97.22,70.48,56.20,41.10,28.56,18.78
2,France,94.76,68.50,43.94,28.62,15.76,8.30
3,Portugal,94.72,67.42,41.74,22.54,12.70,6.80
4,Germany,98.26,69.24,40.62,24.76,12.36,6.56
5,England,96.94,64.10,36.82,19.96,9.96,4.66
6,Brazil,94.86,55.14,32.90,17.76,8.08,3.90
7,Morocco,93.70,53.92,32.18,17.08,7.90,3.56
8,Ecuador,94.98,58.22,29.90,15.92,7.44,3.24
9,Colombia,89.48,55.02,28.92,14.12,7.32,3.00



Champion probabilities sum to: 100.0 %


In [110]:
print("Champion probabilities sum to:",
      round(mc_results["champion_pct"].sum(), 2), "%")

print("\nProbability ranges:")
print(mc_results[
    ["qualification_pct", "round_of_16_pct",
     "quarterfinal_pct", "semifinal_pct",
     "final_pct", "champion_pct"]
].agg(["min", "max"]).round(2))

print("\nTop 10 championship probabilities:")
display(mc_results[["team", "champion_pct"]].head(10))

Champion probabilities sum to: 100.0 %

Probability ranges:
     qualification_pct  round_of_16_pct  quarterfinal_pct  semifinal_pct  \
min               4.88             0.26              0.06           0.00   
max              99.64            73.04             56.20          43.74   

     final_pct  champion_pct  
min        0.0           0.0  
max       31.0          19.8  

Top 10 championship probabilities:


,team,champion_pct
0,Spain,19.80
1,Argentina,18.78
2,France,8.30
3,Portugal,6.80
4,Germany,6.56
5,England,4.66
6,Brazil,3.90
7,Morocco,3.56
8,Ecuador,3.24
9,Colombia,3.00
